# 🚀 Universal Google Drive Uploader

**Upload any file from any URL directly to your Google Drive**

- ✅ No local bandwidth used
- ✅ No local storage needed
- ✅ Supports direct links + video sites (YouTube, Twitter, etc.)
- ✅ Progress tracking

---

## How to Use
1. Run **Cell 1** to mount your Google Drive (one-time per session)
2. Run **Cell 2** to install yt-dlp (for video sites)
3. Enter your URL in **Cell 3** and run it

That's it! File will appear in your Google Drive.

## 📁 Cell 1: Mount Google Drive
Run this once per session. Click the link and authorize access.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted successfully!")

## 📦 Cell 2: Install Dependencies
Run once per session to enable video site support.

In [ ]:
!pip install -q yt-dlp
print("✅ yt-dlp installed (supports 1000+ video sites)")

## ⬆️ Cell 3: Upload File from URL

**Edit the settings below and run this cell:**

In [ ]:
#@title 🔧 Upload Settings
#@markdown ### Enter your URL and destination:

FILE_URL = "https://example.com/file.zip" #@param {type:"string"}
SAVE_FOLDER = "Downloads" #@param {type:"string"}
CUSTOM_FILENAME = "" #@param {type:"string"}

#@markdown ---
#@markdown ### Options:
USE_YTDLP = False #@param {type:"boolean"}
YTDLP_FORMAT = "best" #@param ["best", "bestvideo+bestaudio", "bestaudio", "worst"]

# ============================================================
# DO NOT EDIT BELOW THIS LINE
# ============================================================

import os
import requests
import time
from urllib.parse import urlparse, unquote

def format_size(size_bytes):
    """Convert bytes to human readable format."""
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if size_bytes < 1024.0:
            return f"{size_bytes:.2f} {unit}"
        size_bytes /= 1024.0
    return f"{size_bytes:.2f} PB"

def format_speed(bytes_per_sec):
    """Convert bytes/sec to human readable format."""
    return format_size(bytes_per_sec) + "/s"

def get_filename_from_url(url, response=None):
    """Extract filename from URL or Content-Disposition header."""
    if response and 'Content-Disposition' in response.headers:
        cd = response.headers['Content-Disposition']
        if 'filename=' in cd:
            fname = cd.split('filename=')[1].strip('"\'')
            return unquote(fname)
    
    parsed = urlparse(url)
    path = unquote(parsed.path)
    filename = os.path.basename(path)
    
    if not filename or '.' not in filename:
        return "downloaded_file"
    return filename

def download_direct(url, save_path):
    """Download file directly using requests with progress."""
    print(f"📥 Starting download...")
    print(f"   URL: {url[:80]}..." if len(url) > 80 else f"   URL: {url}")
    
    response = requests.get(url, stream=True, allow_redirects=True)
    response.raise_for_status()
    
    total_size = int(response.headers.get('content-length', 0))
    
    # Get filename
    if CUSTOM_FILENAME:
        filename = CUSTOM_FILENAME
    else:
        filename = get_filename_from_url(url, response)
    
    full_path = os.path.join(save_path, filename)
    
    print(f"   Filename: {filename}")
    if total_size:
        print(f"   Size: {format_size(total_size)}")
    print(f"   Saving to: {full_path}")
    print()
    
    downloaded = 0
    start_time = time.time()
    chunk_size = 8 * 1024 * 1024  # 8MB chunks
    
    with open(full_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=chunk_size):
            if chunk:
                f.write(chunk)
                downloaded += len(chunk)
                
                elapsed = time.time() - start_time
                speed = downloaded / elapsed if elapsed > 0 else 0
                
                if total_size:
                    percent = (downloaded / total_size) * 100
                    print(f"\r   ⬆️ Progress: {percent:.1f}% | {format_size(downloaded)} / {format_size(total_size)} | {format_speed(speed)}", end="")
                else:
                    print(f"\r   ⬆️ Downloaded: {format_size(downloaded)} | {format_speed(speed)}", end="")
    
    elapsed = time.time() - start_time
    avg_speed = downloaded / elapsed if elapsed > 0 else 0
    
    print()
    print()
    print("═" * 50)
    print("✅ UPLOAD COMPLETE!")
    print("═" * 50)
    print(f"   📄 File: {filename}")
    print(f"   📊 Size: {format_size(downloaded)}")
    print(f"   ⏱️ Time: {elapsed:.1f} seconds")
    print(f"   🚀 Speed: {format_speed(avg_speed)}")
    print(f"   📁 Location: {full_path}")
    print("═" * 50)
    
    return full_path

def download_ytdlp(url, save_path):
    """Download using yt-dlp for video sites."""
    import yt_dlp
    
    print(f"🎬 Using yt-dlp for video extraction...")
    print(f"   URL: {url[:80]}..." if len(url) > 80 else f"   URL: {url}")
    print(f"   Format: {YTDLP_FORMAT}")
    print()
    
    # Custom filename template
    if CUSTOM_FILENAME:
        outtmpl = os.path.join(save_path, CUSTOM_FILENAME)
    else:
        outtmpl = os.path.join(save_path, '%(title)s.%(ext)s')
    
    ydl_opts = {
        'format': YTDLP_FORMAT,
        'outtmpl': outtmpl,
        'progress_hooks': [ytdlp_progress_hook],
        'quiet': False,
        'no_warnings': True,
    }
    
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        filename = ydl.prepare_filename(info)
    
    print()
    print("═" * 50)
    print("✅ UPLOAD COMPLETE!")
    print("═" * 50)
    print(f"   📄 Title: {info.get('title', 'Unknown')}")
    print(f"   📁 Location: {filename}")
    print("═" * 50)
    
    return filename

def ytdlp_progress_hook(d):
    """Progress hook for yt-dlp."""
    if d['status'] == 'downloading':
        percent = d.get('_percent_str', 'N/A')
        speed = d.get('_speed_str', 'N/A')
        eta = d.get('_eta_str', 'N/A')
        print(f"\r   ⬆️ {percent} | Speed: {speed} | ETA: {eta}", end="")
    elif d['status'] == 'finished':
        print(f"\n   ✓ Download finished, processing...")

# Main execution
if __name__ == "__main__" or True:
    # Create save directory
    save_path = f"/content/drive/MyDrive/{SAVE_FOLDER}"
    os.makedirs(save_path, exist_ok=True)
    
    print("╔" + "═" * 48 + "╗")
    print("║     UNIVERSAL GOOGLE DRIVE UPLOADER           ║")
    print("╚" + "═" * 48 + "╝")
    print()
    
    try:
        if USE_YTDLP:
            result = download_ytdlp(FILE_URL, save_path)
        else:
            result = download_direct(FILE_URL, save_path)
    except Exception as e:
        print(f"\n❌ Error: {str(e)}")
        print("\n💡 Tips:")
        print("   - For video sites, enable 'USE_YTDLP' option")
        print("   - Check if the URL is accessible")
        print("   - Make sure Google Drive is mounted (run Cell 1)")

---

## 📦 Cell 4: Batch Upload (Multiple URLs)

Upload multiple files at once. Add one URL per line.

In [ ]:
#@title 🔧 Batch Upload Settings
#@markdown ### Enter multiple URLs (one per line):

URLS = """https://example.com/file1.zip
https://example.com/file2.pdf
https://example.com/file3.mp4""" #@param {type:"raw"}

BATCH_SAVE_FOLDER = "BatchDownloads" #@param {type:"string"}
BATCH_USE_YTDLP = False #@param {type:"boolean"}

# ============================================================

import os
import requests
import time
from urllib.parse import urlparse, unquote

# Parse URLs
url_list = [u.strip() for u in URLS.strip().split('\n') if u.strip()]

print("╔" + "═" * 48 + "╗")
print("║         BATCH UPLOAD MODE                      ║")
print("╚" + "═" * 48 + "╝")
print(f"\n📋 Found {len(url_list)} URLs to process\n")

save_path = f"/content/drive/MyDrive/{BATCH_SAVE_FOLDER}"
os.makedirs(save_path, exist_ok=True)

results = []
for i, url in enumerate(url_list, 1):
    print(f"\n{'─' * 50}")
    print(f"📥 [{i}/{len(url_list)}] Processing...")
    print(f"{'─' * 50}")
    
    try:
        CUSTOM_FILENAME = ""  # Reset for each file
        
        if BATCH_USE_YTDLP:
            result = download_ytdlp(url, save_path)
        else:
            result = download_direct(url, save_path)
        
        results.append((url, "✅ Success", result))
    except Exception as e:
        print(f"❌ Failed: {str(e)}")
        results.append((url, "❌ Failed", str(e)))

# Summary
print("\n")
print("═" * 50)
print("📊 BATCH UPLOAD SUMMARY")
print("═" * 50)
success = sum(1 for r in results if "Success" in r[1])
print(f"   ✅ Successful: {success}/{len(url_list)}")
print(f"   ❌ Failed: {len(url_list) - success}/{len(url_list)}")
print(f"   📁 Location: {save_path}")
print("═" * 50)

---

## 🔍 Cell 5: Check Drive Storage

In [ ]:
!df -h /content/drive/MyDrive | tail -1 | awk '{print "📊 Google Drive Storage:", $3, "used /", $2, "total (", $5, "used )"}'

---

## 📂 Cell 6: List Files in Folder

In [ ]:
FOLDER_TO_LIST = "Downloads" #@param {type:"string"}

import os

folder_path = f"/content/drive/MyDrive/{FOLDER_TO_LIST}"

if os.path.exists(folder_path):
    files = os.listdir(folder_path)
    print(f"📂 Contents of '{FOLDER_TO_LIST}':")
    print("─" * 40)
    for f in sorted(files):
        full_path = os.path.join(folder_path, f)
        if os.path.isfile(full_path):
            size = os.path.getsize(full_path)
            print(f"   📄 {f} ({format_size(size)})")
        else:
            print(f"   📁 {f}/")
    print("─" * 40)
    print(f"   Total: {len(files)} items")
else:
    print(f"❌ Folder not found: {folder_path}")